# 03a — Multilayer Perceptron (MLP)

This notebook introduces **numpy** for the first time in this course.

From note 03 onwards, the network is expressed in matrix form:

```
z⁽¹⁾ = W⁽¹⁾ᵀx + b⁽¹⁾
h    = f(z⁽¹⁾)
z⁽²⁾ = W⁽²⁾ᵀh + b⁽²⁾
ŷ    = f(z⁽²⁾)
```

numpy makes this notation concrete — `W1.T @ x + b1` maps directly to `W⁽¹⁾ᵀx + b⁽¹⁾`.

**What this notebook demonstrates:**
1. Defining the MLP weight matrices and verifying their shapes
2. Forward pass — step by step, matching the hand calculation in the note
3. Feature space warping — the (x₁,x₂) → (h₁,h₂) transformation table
4. Why the perceptron rule fails — chain rule expansion on a hidden weight

---
## Part 1: Defining the Network

We use the 2-2-1 MLP from the worked example in note 03.

```
Architecture:  2 inputs → 2 hidden neurons → 1 output
Activation:    Heaviside step  f(z) = 1 if z ≥ 0, else 0
```

Parameters (from the note):
- h₁ learns OR:   w = [1, 1],   b = -0.5
- h₂ learns NAND: w = [-1, -1], b =  1.5
- ŷ  learns AND:  w = [1, 1],   b = -1.5

Weight matrix convention: `wᵢⱼ` = source i → destination j
- Row i of W⁽¹⁾ = weights fanning out from input xᵢ
- Column j of W⁽¹⁾ = weights arriving at hidden neuron hⱼ

In [ ]:
import numpy as np

# Layer 1 weights: shape (2, 2)  — 2 inputs, 2 hidden neurons
# W1[i, j] = weight from input i to hidden neuron j
W1 = np.array([
    [ 1, -1],   # weights from x1: x1→h1=1,  x1→h2=-1
    [ 1, -1],   # weights from x2: x2→h1=1,  x2→h2=-1
])

b1 = np.array([-0.5, 1.5])   # biases for h1 and h2

# Layer 2 weights: shape (2, 1)  — 2 hidden neurons, 1 output
W2 = np.array([
    [1],   # weight from h1 to ŷ
    [1],   # weight from h2 to ŷ
])

b2 = np.array([-1.5])        # bias for output neuron

print('W1 shape:', W1.shape, '  (inputs × hidden neurons)')
print('b1 shape:', b1.shape)
print('W2 shape:', W2.shape, '  (hidden neurons × output neurons)')
print('b2 shape:', b2.shape)
print()
print('W1 =\n', W1)
print('b1 =', b1)
print('W2 =\n', W2)
print('b2 =', b2)

---
## Part 2: Forward Pass — Single Input

We implement the forward pass step by step, matching the equations in the note:

```
z1 = W1.T @ x + b1      ← pre-activations for hidden layer
h  = f(z1)              ← hidden activations
z2 = W2.T @ h + b2      ← pre-activation for output
y_hat = f(z2)           ← final prediction
```

Activation: Heaviside step — `f(z) = 1 if z ≥ 0, else 0`

We trace Case 1 from the note: x = [0, 0], target y = 0.

In [ ]:
def step(z):
    """Heaviside step activation: 1 if z >= 0, else 0."""
    return (z >= 0).astype(int)

# Case 1: x = [0, 0], target y = 0
x = np.array([0, 0])

# Hidden layer
z1 = W1.T @ x + b1
h  = step(z1)

# Output layer
z2    = W2.T @ h + b2
y_hat = step(z2)

print('Case 1: x = [0, 0],  target y = 0')
print()
print('Hidden layer:')
print('  z1 = W1.T @ x + b1 =', z1)
print('  h  = step(z1)       =', h)
print()
print('Output layer:')
print('  z2    = W2.T @ h + b2 =', z2)
print('  y_hat = step(z2)      =', y_hat)
print()
print('Correct:', '✓' if y_hat[0] == 0 else '✗')

---
## Part 3: Forward Pass — All Four XOR Inputs

We now run all four XOR inputs through the network and verify every output matches the hand calculation in the note.

In [ ]:
# All four XOR inputs and targets
inputs  = np.array([[0,0], [0,1], [1,0], [1,1]])
targets = np.array([0, 1, 1, 0])

print('XOR Forward Pass — all four inputs')
print('-' * 65)
print(f'  {"x":>8}  {"z1":>14}  {"h":>8}  {"z2":>8}  {"y_hat":>6}  {"y":>4}  {"ok?":>4}')
print('-' * 65)

for x, y in zip(inputs, targets):
    z1    = W1.T @ x + b1
    h     = step(z1)
    z2    = W2.T @ h + b2
    y_hat = step(z2)[0]
    check = '✓' if y_hat == y else '✗'
    print(f'  {str(x.tolist()):>8}  {str(z1.tolist()):>14}  {str(h.tolist()):>8}  '
          f'{str(z2.tolist()):>8}  {y_hat:>6}  {y:>4}  {check:>4}')

print('-' * 65)
print('All outputs match the hand calculation in note 03. ✓')

---
## Part 4: Feature Space Warping

The hidden layer transforms the input coordinates (x₁, x₂) into a new space (h₁, h₂).

We print the full transformation table and verify the key insight from the note:
both positive XOR inputs — (0,1) and (1,0) — collapse to the same hidden representation (1,1).

In [ ]:
print('Feature Space Warping: (x1, x2) → (h1, h2)')
print('-' * 50)
print(f'  {"input x":>10}  {"z1":>14}  {"hidden h":>10}  {"y":>4}')
print('-' * 50)

for x, y in zip(inputs, targets):
    z1 = W1.T @ x + b1
    h  = step(z1)
    print(f'  {str(x.tolist()):>10}  {str(z1.tolist()):>14}  {str(h.tolist()):>10}  {y:>4}')

print('-' * 50)
print()
print('Key observation:')
print('  (0,1) → h = [1, 1]   (XOR = 1)')
print('  (1,0) → h = [1, 1]   (XOR = 1)  ← same hidden representation')
print()
print('Both positive inputs collapse to the same point in hidden space.')
print('In (h1, h2) space, the problem is linearly separable.')
print('The output neuron only needs: h1 + h2 >= 1.5')

---
## Part 5: Why the Perceptron Rule Fails on Hidden Weights

The perceptron rule is:
```
wᵢ ← wᵢ + η × (y − ŷ) × xᵢ
```

To update the hidden weight w⁽¹⁾₁₁ (from x₁ to h₁), the correct gradient requires the chain rule:

```
∂L/∂w⁽¹⁾₁₁ = dL/dŷ · f'(z⁽²⁾) · w⁽²⁾₁₁ · f'(z₁⁽¹⁾) · x₁
```

We compute both — the naive perceptron update and the correct chain rule gradient — and compare them.

We use Case 2: x = [0, 1], y = 1, and MSE loss L = (y − ŷ)².

In [ ]:
# Case 2: x = [0, 1], y = 1
x = np.array([0, 1])
y = 1
lr = 0.1

# Forward pass — store all intermediate values
z1    = W1.T @ x + b1          # pre-activations: hidden layer
h     = step(z1).astype(float) # hidden activations
z2    = W2.T @ h + b2          # pre-activation: output
y_hat = step(z2)[0]            # final prediction

print('Forward pass for x = [0, 1], y = 1:')
print(f'  z1    = {z1}')
print(f'  h     = {h}')
print(f'  z2    = {z2}')
print(f'  y_hat = {y_hat}')
print()

# ── Perceptron rule (naive) ───────────────────────────────────────────────────
# Applies the output error directly to the hidden weight
error = y - y_hat
perceptron_update_w11 = lr * error * x[0]   # x[0] = x1 = 0

print('Perceptron rule applied naively to w⁽¹⁾₁₁:')
print(f'  error = y - y_hat = {y} - {y_hat} = {error}')
print(f'  Δw⁽¹⁾₁₁ = η × error × x1 = {lr} × {error} × {x[0]} = {perceptron_update_w11}')
print()

# ── Chain rule (correct gradient) ────────────────────────────────────────────
# L = (y - y_hat)²  →  dL/dy_hat = -2(y - y_hat) = 2(y_hat - y)
# f'(z) for step function = 0 everywhere (undefined at 0, treated as 0)
# We use a soft approximation here to show the structure — in practice
# this is why differentiable activations (note 04) are required

dL_dyhat  = 2 * (y_hat - y)          # ∂L/∂ŷ  (MSE gradient)
w21_11    = W2[0, 0]                  # w⁽²⁾₁₁ = weight from h1 to output
x1        = x[0]                      # x1 = 0

print('Chain rule expansion for ∂L/∂w⁽¹⁾₁₁:')
print(f'  dL/dŷ          = 2(ŷ - y)   = 2({y_hat} - {y}) = {dL_dyhat}')
print(f"  f'(z⁽²⁾)       = f'({z2[0]:.1f})     ← derivative of step = 0 (needs smooth activation)")
print(f'  w⁽²⁾₁₁         = {w21_11}              ← weight in layer above')
print(f"  f'(z₁⁽¹⁾)      = f'({z1[0]:.1f})     ← derivative of step = 0 (needs smooth activation)")
print(f'  x₁             = {x1}              ← input value')
print()
print('What the perceptron rule has access to:')
print(f'  ✓  error (y - ŷ) = {error}')
print(f'  ✓  x1            = {x1}')
print()
print('What the perceptron rule cannot see:')
print(f"  ✗  f'(z⁽²⁾)  — derivative of output activation")
print(f'  ✗  w⁽²⁾₁₁   — weight in the layer above')
print(f"  ✗  f'(z₁⁽¹⁾) — derivative of hidden activation")
print()
print('The perceptron rule is structurally blind to 3 of the 5 required factors.')
print('This is the credit assignment problem — solved by backpropagation (note 06).')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| Weight matrix shapes | W1: (2,2), b1: (2,), W2: (2,1), b2: (1,) |
| Forward pass | `z1 = W1.T @ x + b1`, `h = step(z1)`, `z2 = W2.T @ h + b2` |
| All 4 XOR inputs | Every output matches the hand calculation in note 03 |
| Feature space warping | (0,1) and (1,0) both map to h = [1,1] — problem becomes linearly separable |
| Perceptron rule failure | Rule has access to 2 of 5 required chain rule factors — structurally blind to the rest |

**numpy introduced here because:**
- Note 03 expresses the forward pass in matrix form: `z = Wᵀx + b`
- `W1.T @ x + b1` maps directly to the notation in the note
- All future notes build on this matrix form

**Next:** Note 04 introduces differentiable activation functions — the first of the three pillars needed to train hidden weights.